In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, f1_score
%pip install kagglehub catboost xgboost -q



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

data = pd.read_csv(path + "/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:

data.head()

In [ ]:
# Task 3: Write your code here:

data.info()

In [ ]:
# Task 4: Write your code here:

data.describe()
data.duplicated()

In [ ]:
# Task 1: Write your code here:

data.isnull().sum()
data.drop("D_142", axis=1, inplace=True)

cols = data.isnull().sum().index
for col in cols:
    data[col].fillna(data[col].mean(), inplace=True)

data.isnull().sum().sum()
data


In [ ]:
# Task 2: Write your code here:

if not data.duplicated().sum():
  print("No duplicates. MOVE ON")
else:
  print("Removing duplicates")

In [ ]:
# Task 3: Write your code here:

cat_cols = data.select_dtypes(include="object").columns
cat_cols

In [ ]:
# Task 4: Write your code here:

num_cols = data.select_dtypes(exclude="object").columns

for col in num_cols[:-1]:
  ss = StandardScaler()
  data[col] = ss.fit_transform(pd.DataFrame(data[col]))


In [ ]:
# Task 5: Write your code here:

data["Target"].hist(bins=30, edgecolor='black')
plt.xlabel("Target")
plt.ylabel("Frequency")
plt.title("Target Distribution")
plt.show()

In [ ]:
# Task 1: Write your code here:

X = data.drop("Target", axis=1)
y = data["Target"]


In [ ]:
# Task 2,3,4,5: Write your code here:
from catboost import CatBoostClassifier

f1_list = []

skf = StratifiedKFold(n_splits=5, shuffle=True)
model = CatBoostClassifier()

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{5}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn model
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  f1 = f1_score(y_test, y_pred, zero_division=0)

  f1_list.append(f1)

avg_loss = np.mean(f1_list, axis=0)
print(avg_loss)


In [ ]:
# Task 1: Write your code here:

feature_cols = X.columns

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

print(f"The Golden Feature is {feature_importance['feature'].max()}")

In [ ]:
# Task Bonus: Write your code here: